In [5]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

c:\Users\LENONO\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading all PDFS

In [19]:
def load_all_pdfs(path_pdf):
    all_docs=[]
    pdf_dir=Path(path_pdf)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} docs in the folder")

    for pdf_file in pdf_files:
        try:
            print(f"processing : {pdf_file.name}")
            loader=PyMuPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            all_docs.extend(documents)
        except Exception as e:
            print(f"error processing files : {e}")
            raise
    
    print(f"loaded {len(all_docs)} documents ")
    return all_docs

In [20]:
all_pdf_files=load_all_pdfs("data/")
all_pdf_files

found 1 docs in the folder
processing : LOS_Question_Bank.pdf
loaded 1 documents 


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': 'data\\pdf\\LOS_Question_Bank.pdf', 'file_path': 'data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1\nDefine a signal in Linux. List any four commonly used signals.\n2\nExplain the purpose of signal() system call.\n3\nWhat is sigaction()? How is it better than signal()?\n4\nDifferentiate between kill() and raise().\n5\nExplain thread and its advantages.\n6\nExplain pthread_create() and pthread_join().\n7\nWhat is mute

Chunking

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [30]:
def split_into_chunks(documents,chunk_size=300,chunk_overlap=60):
    splitter=RecursiveCharacterTextSplitter(
        separators=["\n\n","\n"," ",""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    split_docs=splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content : {split_docs[0].page_content[:200]}")
        print(f"metadata : {split_docs[0].metadata}")
    return split_docs

In [31]:
chunks=split_into_chunks(all_pdf_files)
chunks

split 1 documents into 5 chunks
Content : Linux Operating Systems (LOS)
Question Bank – 5 & 10 Marks
5 MARK QUESTIONS
1
Define a signal in Linux. List any four commonly used signals.
2
Explain the purpose of signal() system call.
3
What is si
metadata : {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': 'data\\pdf\\LOS_Question_Bank.pdf', 'file_path': 'data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': 'data\\pdf\\LOS_Question_Bank.pdf', 'file_path': 'data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1\nDefine a signal in Linux. List any four commonly used signals.\n2\nExplain the purpose of signal() system call.\n3\nWhat is sigaction()? How is it better than signal()?\n4\nDifferentiate between kill() and raise().\n5'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecif

Initializing the Embedding Model

In [32]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import uuid
from typing import List,Dict,Any

In [49]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Failed to load model : {e}")
            raise
    
    def generate_embeddings(self,texts:List[Any])->np.ndarray:
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings of shape : {embeddings.shape}")
        return embeddings
    

In [50]:
embedding_manager=EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3955.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


In [51]:
embedding_manager

We store to the vector DB now

In [52]:
import os

In [61]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents_self",persistent_dir="../data/vector_store_self"):
        self.collection_name=collection_name
        self.persistent_dir=persistent_dir
        self.collection=None
        self.client=None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persistent_dir)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={'desc':"pdf embeddings for rag"}
            )
            print(f"vector store initialized : {self.collection_name}")
            print(f"existing documents in the vector store : {self.collection.count()}")

        except Exception as e:
            print(f"error initializing the DB : {e}")
            raise 
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Make sure the document size and the embedding size are the same")
        
        ids=[]
        metadatas=[]
        document_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_lentth']=len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                embeddings=embeddings_list,
                documents=document_text
            )
            print(f"added {len(documents)} documents to the vector store")
        except Exception as e:
            print(f"an error occurred while adding : {e}")
            raise 


In [62]:
vectorstore=VectorStore()

vector store initialized : pdf_documents_self
existing documents in the vector store : 0


In [63]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': 'data\\pdf\\LOS_Question_Bank.pdf', 'file_path': 'data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1\nDefine a signal in Linux. List any four commonly used signals.\n2\nExplain the purpose of signal() system call.\n3\nWhat is sigaction()? How is it better than signal()?\n4\nDifferentiate between kill() and raise().\n5'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecif

In [64]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.14it/s]

Generated embeddings of shape : (5, 384)
added 5 documents to the vector store
